1.- Reading from local source JSON --> Bornze Ingestion

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE `1_bronze_db`.customer_bronze_raw
  COMMENT "Raw data from customes CDC feed"
  TBLPROPERTIES(
    "quality" = "bronze",
    "pipelines.reset.allowed" = false -- prevent full table refreshes on the bronze table
  )
  AS
select 
*, 
CURRENT_TIMESTAMP() processing_time, -- Obtain the ingestion processing for the rows
_metadata.file_name as source_file -- Obtain the file name of the record
 from STREAM read_files(
  concat(:source_vol,"/customer1.json"), -- source = "/Volumes/my_catalog/my_schema"
  format => "json",
  multiLine => 'true'
)  


2.- Crear the Bronze Clean Straeaming Table with Data Quality Enforcement

In [0]:
CREATE STREAMING TABLE IF NOT EXISTS `1_bronze_db`.customer_bronze_clean
 ( 
  CONSTRAINT valid_id EXPECT(customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
  CONSTRAINT valid_operation EXPECT (operation is not null) on violation drop ROW,
  CONSTRAINT valid_name EXPECT (name IS NOT NULL OR operation ="DELETE"), --WARNING CONSTRAIN
  CONSTRAINT valid_address EXPECT (
    (address IS NOT NULL and 
    city IS NOT NULL and 
    state IS NOT NULL AND 
    zip_code IS NOT NULL) OR operation ="DELETE"),
    CONSTRAINT valid_email EXPECT
        ( 
            rlike(email, '^([a-zA-Z0-9_\\-\\.]+)@[a-zA-Z0-9_\\-\\.]+\\.([a-zA-Z]{2,4})$')
            OR operation ="DELETE"
        ) on violation DROP ROW
        
)
COMMENT "Cleaned RAW BRONZE TIMESTAMP COLUMN AND DATA QUALITY CONSTRAINTS"

AS SELECT 
*,
CAST(from_unixtime(timestamp) AS timestamp) AS timestamp_datetime
 FROM STREAM `1_bronze_db`.customer_bronze_raw;



- Processing CDC Data with AUTO CDC INTO

In [0]:
CREATE OR REFRESH STREAMING TABLE `2_silver_db`.scd_type_1_customers_silver
  COMMENT 'SCD Type 1 Historical Customer Data';


In [0]:
CREATE FLOW scd_type_1_flow AS
AUTO CDC INTO  `2_silver_db`.scd_type_1_customers_silver --TARGET TABLE TO UPDATE WITH scd tYPE 1(OR 2)
FROM STREAM `1_bronze_db`.customer_bronze_clean -- Source records to determine updates, deletes and inserts
KEYS (customer_id)                         -- Primary key for identifying records
APPLY AS DELETE WHEN operation = "DELETE"  -- Handle deletes from source to the target
SEQUENCE BY timestamp_datetime             -- Defines order of operations for applying changes
COLUMNS * EXCEPT (timestamp, _rescued_date, operation)  --Select columns and exclude metada fields
STORED AS SCD TYPE 1;                      -- Use SCD type 1 TO update the target table (no historical information)